## Day5 Challenge - Company Brochure Visual Generator - English & Spanish
## Create a visual design in HTML

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
from bs4 import BeautifulSoup
import requests
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit

    """
    
 
    response = requests.get(url, headers=headers, verify=False)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers, verify=False)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]

In [ ]:
load_dotenv(override=True)

# Create OpenAI client
# nice simple wrapper around making an HTTP call to an API cloud., it looks for gemini key by default,

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("GOOGLE_API_KEY")

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
MODEL="gemini-3.1-flash-lite"



## Prompt 1 - for finding relevant links

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

## First function for selecting relevant links using LLM

In [6]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model= MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

## Second Function for extracting contents of landing page and all relevant links

In [7]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
        print(result)
    return result

## Second Prompt for creating brochure. Add instruction to output HTML code that can be opened in a browser.

In [8]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
output the response in html format,that can further be opened in a webbrowser. Include details of company culture, customers and careers/jobs if you have the information.

"""

In [9]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages; Include details of company culture, customers and careers/jobs if you have the information.
use this information to build a short brochure of the company, output the response in html format,that can further be opened in a webbrowser.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

## Third Function to create brochure for the company. 

In [10]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model= MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

## Overall flow looks like below-
1) Using company name and URL, It extracted contents of landing page using scraping - fetch website contents 
2) It extracted all links on landing page using scraping - fetch website links. 
3) Out of all links, it identified relevant links using LLM call.
4) For each relevant link - it extracted contents of the page using scraping - fetch website contents.
5) Using overall extracted content, it created brochure using LLM another call.

There are two LLM calls in total - one for identifying relevant links and another for creating Brochure. 

In [11]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-5.2
Updated
about 11 hours ago
•
40.1k
•
2.13k
yuxinlu1/gemma-4-12B-coder-fable5-composer2.5-v1-GGUF
Updated
4 days ago
•
456k
•
2.21k
WeiboAI/VibeThinker-3B
Updated
4 days ago
•
41.2k
•
645
yuxinlu1/gemma-4-12B-agentic-fable5-composer2.5-v2-3.5x-tau2-GGUF
Updated
4 days 

```html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Hugging Face: The Home of Machine Learning</title>
    <style>
        body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; line-height: 1.6; color: #333; max-width: 800px; margin: 0 auto; padding: 20px; background-color: #f9f9f9; }
        header { text-align: center; padding: 40px 0; background: #fff; border-bottom: 3px solid #ffcc00; }
        h1 { color: #1a1a1a; }
        .section { background: #fff; padding: 25px; margin-bottom: 20px; border-radius: 8px; box-shadow: 0 2px 5px rgba(0,0,0,0.1); }
        h2 { color: #ff8800; }
        .highlight { font-weight: bold; }
    </style>
</head>
<body>

<header>
    <h1>Hugging Face</h1>
    <p><em>The AI community building the future.</em></p>
</header>

<div class="section">
    <h2>About Us</h2>
    <p>Hugging Face is the leading platform for the machine learning community to create, discover, and collaborate. Our mission is to <strong>democratize good machine learning</strong>, one commit at a time. We serve as the central hub where researchers, developers, and organizations come together to build the next generation of AI.</p>
</div>

<div class="section">
    <h2>Our Platform</h2>
    <p>We provide the infrastructure and tools that power the modern AI ecosystem, including:</p>
    <ul>
        <li><strong>Models:</strong> Access to over 2 million machine learning models.</li>
        <li><strong>Datasets:</strong> A vast repository of over 500,000 datasets for training and testing.</li>
        <li><strong>Spaces:</strong> A platform to host and showcase interactive AI applications.</li>
        <li><strong>Enterprise Solutions:</strong> Robust infrastructure, including Inference Endpoints, storage buckets, and professional support for scaling AI in business environments.</li>
    </ul>
</div>

<div class="section">
    <h2>Company Culture</h2>
    <p>Our culture is rooted in <strong>open-source values and global collaboration</strong>. We believe in transparency and community-led innovation. As an organization, we actively publish research, blog about the state of the open-source ecosystem, and facilitate learning through our dedicated tracks and community forums. We are a collaborative, mission-driven team that values sharing knowledge and building technology that serves the global community.</p>
</div>

<div class="section">
    <h2>Customers & Community</h2>
    <p>Hugging Face is the preferred platform for both independent developers and large-scale enterprises. Our customers range from AI startups to global technology organizations who rely on our platform to host models, manage data, and deploy AI agents at scale. Our community is our greatest strength, with over 100,000 followers and daily contributions from thousands of developers worldwide.</p>
</div>

<div class="section">
    <h2>Careers</h2>
    <p>We are looking for passionate individuals who want to help shape the future of AI. Whether you are an engineer, a researcher, or an advocate for open-source, joining Hugging Face means working at the bleeding edge of machine learning. You will work alongside a diverse, global team dedicated to building sustainable and accessible AI technology.</p>
    <p><strong>Join us:</strong> Visit our <a href="https://huggingface.co">official website</a> to see our active projects, follow our updates, and find out how to contribute to the future of AI.</p>
</div>

<footer>
    <p style="text-align: center; font-size: 0.8em; color: #777;">&copy; 2025 Hugging Face. Building the future of AI together.</p>
</footer>

</body>
</html>
```

## Get Brochure in Spanish

In [12]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits. Translate all english language contents to Spanish.
output the response in html format,that can further be opened in a webbrowser. Include details of company culture, customers and careers/jobs if you have the information.

"""

In [13]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages; Include details of company culture, customers and careers/jobs if you have the information.
Translate all english language contents to Spanish. use this information to build a short brochure of the company, output the response in html format,that can further be opened in a webbrowser.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [14]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model= MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [15]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 6 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-5.2
Updated
about 11 hours ago
•
40.1k
•
2.13k
yuxinlu1/gemma-4-12B-coder-fable5-composer2.5-v1-GGUF
Updated
4 days ago
•
456k
•
2.21k
WeiboAI/VibeThinker-3B
Updated
4 days ago
•
41.2k
•
645
yuxinlu1/gemma-4-12B-agentic-fable5-composer2.5-v2-3.5x-tau2-GGUF
Updated
4 days 

Aquí tienes el folleto informativo de Hugging Face en formato HTML, traducido al español.

```html
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Hugging Face - Construyendo el Futuro de la IA</title>
    <style>
        body { font-family: sans-serif; line-height: 1.6; color: #333; max-width: 800px; margin: auto; padding: 20px; background-color: #f4f7f6; }
        header { text-align: center; padding: 20px; background: #ff9d00; color: white; border-radius: 8px; }
        section { background: white; padding: 20px; margin: 20px 0; border-radius: 8px; box-shadow: 0 2px 5px rgba(0,0,0,0.1); }
        h2 { color: #ff9d00; }
        .footer { text-align: center; font-size: 0.9em; color: #777; }
    </style>
</head>
<body>

    <header>
        <h1>Hugging Face</h1>
        <p>La comunidad de IA construyendo el futuro.</p>
    </header>

    <section>
        <h2>¿Quiénes somos?</h2>
        <p>Hugging Face es la plataforma de colaboración líder para la comunidad de aprendizaje automático (Machine Learning). Nuestra misión es <strong>democratizar el buen aprendizaje automático</strong>, un commit a la vez.</p>
        <p>Actuamos como el centro global donde investigadores, desarrolladores y empresas se reúnen para descubrir, compartir y colaborar en modelos de IA, conjuntos de datos (datasets) y aplicaciones innovadoras.</p>
    </section>

    <section>
        <h2>Cultura y Comunidad</h2>
        <p>La cultura de Hugging Face está profundamente arraigada en el código abierto y la transparencia. Somos el hogar de una vibrante comunidad global donde:</p>
        <ul>
            <li><strong>Colaboramos:</strong> Más de 2 millones de modelos y 500,000 conjuntos de datos están disponibles para la comunidad.</li>
            <li><strong>Aprendemos:</strong> Fomentamos el intercambio de conocimientos a través de blogs, artículos técnicos y foros activos.</li>
            <li><strong>Innovamos:</strong> Apoyamos el ecosistema abierto, permitiendo que cualquier persona contribuya a los avances más recientes de la Inteligencia Artificial.</li>
        </ul>
    </section>

    <section>
        <h2>Clientes y Soluciones Empresariales</h2>
        <p>Hugging Face ofrece soluciones robustas para organizaciones que buscan integrar la IA en sus flujos de trabajo. Nuestros servicios incluyen:</p>
        <ul>
            <li><strong>Enterprise Support & Inference:</strong> Proporcionamos endpoints de inferencia, proveedores de inferencia y soporte dedicado para empresas.</li>
            <li><strong>Colaboración a escala:</strong> Herramientas para que los equipos trabajen de forma segura en sus propios proyectos de IA dentro de la plataforma.</li>
            <li><strong>Infraestructura:</strong> Almacenamiento seguro (Storage Buckets) y herramientas para el despliegue de aplicaciones de IA en producción.</li>
        </ul>
    </section>

    <section>
        <h2>Carreras: Únete a nosotros</h2>
        <p>En Hugging Face, estamos buscando personas apasionadas por el futuro de la tecnología. Si crees en la democratización del acceso a la IA y quieres ser parte de un equipo que construye la infraestructura del mañana, ¡queremos conocerte!</p>
        <p>Formar parte de nuestro equipo significa trabajar en la intersección de la investigación de vanguardia y la ingeniería de software a escala global.</p>
    </section>

    <div class="footer">
        <p>Descubre más en <a href="https://huggingface.co">huggingface.co</a></p>
    </div>

</body>
</html>
```